<a href="https://colab.research.google.com/github/LokmindTech/TMDB-p-1/blob/main/02_sql_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install mysql-connector-python

In [ ]:
from sqlalchemy import create_engine, URL
from getpass import getpass

In [ ]:
url = URL.create(
    "mysql+mysqlconnector",
    username="ap79ShHiJnSQWTa.root",
    password="3hr30YnebLaqaA89",
    host="gateway01.ap-southeast-1.prod.aws.tidbcloud.com",
    port=4000,
    database="TMDB"
)

In [ ]:
engine = create_engine(url)

In [ ]:
import pandas as pd
from pathlib import Path

folder_path = Path("/content/drive/MyDrive/Cleaned data")

files = {
    "movies": " movies.csv",
    "genres": " genres.csv",
    "movie_genres": " movie_genres.csv",
    "cast": " cast.csv",
    "crew": " crew.csv",
    "movie_keywords": " movie_keywords.csv"
}

for table_name, filename in files.items():
  df = pd.read_csv(folder_path / filename)

  df.to_sql(
            name=table_name,
            con=engine,
            if_exists="replace",
            index=False,
        )

  print(f"{table_name}: {len(df)} rows uploaded")

print("All six datasets uploaded successfully!")

movies: 2503 rows uploaded
genres: 19 rows uploaded
movie_genres: 6878 rows uploaded
cast: 24902 rows uploaded
crew: 7289 rows uploaded
movie_keywords: 41681 rows uploaded
All six datasets uploaded successfully!


# SQL Analysis — 10 Questions

**1) List all movies in the Action genre released after 2015, showing title and release date.**

In [ ]:
import pandas as pd

query1 = """
SELECT DISTINCT m.title, m.release_date
FROM movies m
JOIN movie_genres mg
    ON m.movie_id = mg.movie_id
JOIN genres g
    ON mg.genre_id = g.genre_id
WHERE LOWER(g.genre_name) = 'action'
  AND m.release_date >= '2016-01-01'
ORDER BY m.release_date;
"""

with engine.connect() as i:
    q1 = pd.read_sql_query(query1, i)

display(q1)

,title,release_date
0,The 5th Wave,2016-01-14
1,Kung Fu Panda 3,2016-01-23
2,Zoolander 2,2016-02-06
3,Deadpool,2016-02-09
4,Triple 9,2016-02-19
...,...,...
263,F1,2025-06-25
264,Superman,2025-07-09
265,The Fantastic 4: First Steps,2025-07-23
266,Predator: Badlands,2025-11-05


**2) Find the top 10 highest-grossing movies along with the genre(s) each belongs to.**

In [ ]:
query2 = """
SELECT title, revenue
FROM movies
WHERE revenue > 0
ORDER BY revenue DESC
LIMIT 10;
"""

with engine.connect() as i:
    q2 = pd.read_sql_query(query2, i)

display(q2)

,title,revenue
0,Avatar,2.923706e+09
1,Avengers: Endgame,2.799439e+09
2,Avatar: The Way of Water,2.334485e+09
3,Titanic,2.264162e+09
4,Star Wars: The Force Awakens,2.068224e+09
5,Avengers: Infinity War,2.052415e+09
6,Spider-Man: No Way Home,1.921207e+09
7,Zootopia 2,1.868209e+09
8,Inside Out 2,1.698864e+09
9,Jurassic World,1.671537e+09


**3) Calculate the average budget and average revenue for each genre, sorted by average revenue descending.**

In [ ]:
query3 = """
SELECT
    g.genre_name,
    AVG(NULLIF(m.budget, 0)) AS avg_budget,
    AVG(NULLIF(m.revenue, 0)) AS avg_revenue
FROM movies m
JOIN movie_genres mg ON m.movie_id = mg.movie_id
JOIN genres g ON mg.genre_id = g.genre_id
GROUP BY g.genre_id, g.genre_name
ORDER BY avg_revenue DESC;
"""

with engine.connect() as conn:
    q3 = pd.read_sql_query(query3, conn)

display(q3)

,genre_name,avg_budget,avg_revenue
0,Adventure,1.106736e+08,4.003113e+08
1,Animation,8.763551e+07,3.718357e+08
2,Family,8.790630e+07,3.533691e+08
3,Science Fiction,9.672225e+07,3.336488e+08
4,Action,9.422751e+07,3.084532e+08
5,Fantasy,8.864007e+07,3.027060e+08
6,Comedy,5.450018e+07,2.215264e+08
7,Music,4.281702e+07,2.015662e+08
8,Romance,3.693970e+07,1.748110e+08
9,Thriller,5.121405e+07,1.680042e+08


**4) List the top 10 actors who have appeared in the most movies in the dataset.(cast)**

In [ ]:
query4 = """
SELECT
    actor_name,
    COUNT(DISTINCT movie_id) AS movie_count
FROM `cast`
GROUP BY actor_name
ORDER BY movie_count DESC
LIMIT 10;
"""

with engine.connect() as conn:
    q4 = pd.read_sql_query(query4, conn)

display(q4)

,actor_name,movie_count
0,Samuel L. Jackson,39
1,Robert De Niro,38
2,Brad Pitt,38
3,Johnny Depp,37
4,Tom Hanks,33
5,Scarlett Johansson,32
6,Mark Wahlberg,32
7,Willem Dafoe,32
8,Tom Cruise,31
9,Morgan Freeman,31


**5) Find all directors who have directed more than 3 movies, along with their movie count.**

In [ ]:
query5 = """
SELECT
    person_name AS director_name,
    COUNT(DISTINCT movie_id) AS movie_count
FROM crew
WHERE LOWER(TRIM(job)) = 'director'
GROUP BY person_name
HAVING COUNT(DISTINCT movie_id) > 3
ORDER BY movie_count DESC;
"""

with engine.connect() as conn:
    q5 = pd.read_sql_query(query5, conn)

display(q5)

,director_name,movie_count
0,Steven Spielberg,27
1,Ridley Scott,19
2,Tim Burton,19
3,Martin Scorsese,16
4,Michael Bay,15
...,...,...
212,Charlie Chaplin,4
213,Guy Hamilton,4
214,Pete Docter,4
215,Ryan Coogler,4


**6) List the top 10 most frequently used keywords across all movies.**

In [ ]:
query6 = """
SELECT
    keyword_name,
    COUNT(DISTINCT movie_id) AS movie_count
FROM movie_keywords
WHERE keyword_name IS NOT NULL
GROUP BY keyword_name
ORDER BY movie_count DESC
LIMIT 10;
"""

with engine.connect() as conn:
    q6 = pd.read_sql_query(query6, conn)

display(q6)

,keyword_name,movie_count
0,based on novel or book,449
1,sequel,416
2,duringcreditsstinger,296
3,aftercreditsstinger,242
4,murder,176
5,based on true story,166
6,new york city,161
7,villain,160
8,3d animation,155
9,based on comic,154


**7) Find movies where the budget was above the overall average budget but the revenue was below the overall average revenue.**

In [ ]:
query7 = """
SELECT title, budget, revenue
FROM movies
WHERE budget > 0
  AND revenue > 0
  AND budget > (
      SELECT AVG(NULLIF(budget, 0))
      FROM movies
  )
  AND revenue < (
      SELECT AVG(NULLIF(revenue, 0))
      FROM movies
  )
ORDER BY budget DESC;
"""

with engine.connect() as conn:
    q7 = pd.read_sql_query(query7, conn)

display(q7)

,title,budget,revenue
0,The Marvels,274800000.0,206136825.0
1,Red One,250000000.0,185700759.0
2,Wake Up Dead Man: A Knives Out Mystery,210000000.0,4000000.0
3,The Tomorrow War,200000000.0,14400000.0
4,Onward,200000000.0,141888276.0
...,...,...,...
268,Rambo III,63000000.0,189015611.0
269,Æon Flux,62000000.0,53321673.0
270,The Equalizer 2,62000000.0,190400157.0
271,2 Guns,61000000.0,131940411.0


**8) List all actors who have appeared in a movie directed by a specific director (e.g., Christopher Nolan).**

In [ ]:
query8 = """
SELECT DISTINCT c.actor_name
FROM `cast` c
JOIN crew cr ON c.movie_id = cr.movie_id
WHERE LOWER(TRIM(cr.person_name)) = 'christopher nolan'
  AND LOWER(TRIM(cr.job)) = 'director'
ORDER BY c.actor_name;
"""

with engine.connect() as conn:
    q8 = pd.read_sql_query(query8, conn)

display(q8)

,actor_name
0,Aaron Eckhart
1,Al Pacino
2,Alon Aboutboul
3,Andy Serkis
4,Aneurin Barnard
...,...
79,Tom Hardy
80,Tom Wilkinson
81,Topher Grace
82,Wes Bentley


**9) Find the genre with the highest average rating, considering only genres with at least 20 movies.**


In [ ]:
query9 = """
SELECT
    g.genre_name,
    AVG(m.vote_average) AS avg_rating,
    COUNT(DISTINCT m.movie_id) AS movie_count
FROM movies m
JOIN movie_genres mg ON m.movie_id = mg.movie_id
JOIN genres g ON mg.genre_id = g.genre_id
WHERE m.vote_average IS NOT NULL
GROUP BY g.genre_id, g.genre_name
HAVING COUNT(DISTINCT m.movie_id) >= 20
ORDER BY avg_rating DESC
LIMIT 1;
"""

with engine.connect() as conn:
    q9 = pd.read_sql_query(query9, conn)

display(q9)

,genre_name,avg_rating,movie_count
0,War,7.357576,85


**10) List the top 10 movies with the highest number of keywords tagged to them.**

In [ ]:
import pandas as pd

query10 = """
SELECT
    m.title,
    COUNT(DISTINCT mk.keyword_id) AS keyword_count
FROM movies m
JOIN movie_keywords mk
    ON m.movie_id = mk.movie_id
GROUP BY m.movie_id, m.title
ORDER BY keyword_count DESC
LIMIT 10;
"""

with engine.connect() as conn:
    q10 = pd.read_sql_query(query10, conn)

display(q10)

,title,keyword_count
0,Smile 2,98
1,Taken 3,70
2,Silent Hill,63
3,Twelve Monkeys,57
4,War for the Planet of the Apes,54
5,The Shining,52
6,Dawn of the Planet of the Apes,51
7,How to Train Your Dragon,50
8,How to Train Your Dragon: The Hidden World,49
9,13 Hours: The Secret Soldiers of Benghazi,49
